# Восстановление параметров распределения для региональных данных

In [3]:
import os
import pandas as pd
from tqdm import tqdm

## 1. Подготовка таблицы с данными

In [4]:
POVERTY_FILE = './data/poverty.xlsx'
LINE_FILE = './data/poverty_line.xlsx'
INCOME_FILE = './data/average_income.xlsx'
BASKET_FILE = './data/regional_basket.xls'

In [5]:
basket = pd.read_excel(BASKET_FILE, sheet_name='Данные', skiprows=3, dtype='string')

In [6]:
month_columns = ['январь', 'февраль', 'март', 'апрель', 'май', 'июнь', 'июль', 'август', 'сентябрь', 'октябрь', 'ноябрь', 'декабрь']
basket['average'] = basket[month_columns].astype(float).mean(axis=1)
basket['rate'] = basket['average'] / basket.loc[0, 'average']
basket['Unnamed: 0'] = basket['Unnamed: 0'].str.strip()
basket = basket.rename(columns={'Unnamed: 0': 'region'})

In [7]:
region_replace_dict = {
    'Российская Федерация без учета новых субъектов (с 01.01.2023)': 'Российская Федерация',
    'Архангельская область (кроме Ненецкого автономного округа)': 'Архангельская область без  авт. округа',
    'Город Москва столица Российской Федерации город федерального значения': 'Город Москва',
    'Город Санкт-Петербург город федерального значения': 'Город Санкт-Петербург',
    'Город федерального значения Севастополь': 'Город Севастополь',
    'Кемеровская область - Кузбасс': 'Кемеровская область',
    'Ненецкий автономный округ (Архангельская область)': 'Ненецкий автономный округ',
    'Республика Адыгея (Адыгея)': 'Республика Адыгея',
    'Республика Татарстан (Татарстан)': 'Республика Татарстан',
    'Тюменская область (кроме Ханты-Мансийского автономного округа-Югры и Ямало-Ненецкого автономного округа)': 'Тюменская область (без авт. округов)',
    'Ханты-Мансийский автономный округ - Югра (Тюменская область)': 'Ханты-Мансийский автономный округ - Югра',
    'Чувашская Республика - Чувашия': 'Чувашская Республика',
    'Ямало-Ненецкий автономный округ (Тюменская область)': 'Ямало-Ненецкий автономный округ',
}
basket['region'] = basket['region'].replace(region_replace_dict, regex=False)

In [8]:
poverty = pd.read_excel(POVERTY_FILE, sheet_name='по РФ и субъектам РФ 1995-2024', skiprows=2, dtype='string')
poverty_line = pd.read_excel(LINE_FILE, sheet_name='по субъектам РФ 2024', skiprows=3, dtype='string')
income = pd.read_excel(INCOME_FILE, sheet_name='Лист1', skiprows=2, dtype='string')
regions_codes = pd.read_csv('./data/regions_codes.csv', 
                            dtype='string', 
                            encoding='utf-8', 
                            sep=',').rename(columns={'Код региона': 'region_code', 
                                                     'Название региона': 'region',
                                                     'Короткое название': 'region_short_name'})

In [9]:
regions_codes['region'] = regions_codes['region'].map({
    'Архангельская область': 'Архангельская область без  авт. округа',
    'Тюменская область': 'Тюменская область (без авт. округов)',
    'Москва': 'Город Москва',
    'Санкт-Петербург': 'Город Санкт-Петербург',
    'Севастополь': 'Город Севастополь',
    'Ханты-Мансийский автономный округ': 'Ханты-Мансийский автономный округ - Югра'
}).fillna(regions_codes['region'])
regions_codes.loc[len(regions_codes)] = ['67', 'Город Севастополь', 'Севастополь']
regions_codes.loc[len(regions_codes)] = ['00', 'Российская Федерация', 'Россия']


In [10]:
poverty = poverty[['Unnamed: 0', 2024]][poverty[2024].notna()]
poverty =poverty.rename(columns={'Unnamed: 0': 'region', 2024: 'poverty'})
poverty['region'] = poverty['region'].str.replace('в том числе:', '', regex=False).str.replace('\n', '', regex=False).str.strip()

In [11]:
poverty_line = poverty_line[['Unnamed: 0', 'Unnamed: 1']][poverty_line['Unnamed: 1'].notna()]
poverty_line = poverty_line.rename(columns={'Unnamed: 0': 'region', 'Unnamed: 1': 'poverty_line'})
poverty_line['region'] = poverty_line['region'].str.replace('в том числе:', '', regex=False).str.replace('\n', '', regex=False).str.strip()

poverty_line['region'] = poverty_line['region'].replace({
    'Ханты-Мансийский авт. округ - Югра': 'Ханты-Мансийский автономный округ - Югра',
    'Тюменская область без авт. округов': 'Тюменская область (без авт. округов)',
    'Архангельская область без авт. округа': 'Архангельская область без  авт. округа'
})

In [12]:
income = income[['Unnamed: 0', 'Unnamed: 66']][(income['Unnamed: 66'].notna()) & (income['Unnamed: 66'] != 'год')]
income = income.rename(columns={'Unnamed: 0': 'region', 'Unnamed: 66': 'income'})
income['region'] = income['region'].str.replace('в том числе:', '', regex=False).str.replace('\n', '', regex=False).str.strip()

income['region'] = income['region'].str.replace('г. ', 'Город ', regex=False)
income['region'] = income['region'].str.replace('  ', ' ', regex=False)

income['region'] = income['region'].replace({
    'Архангельская область без авт.округа': 'Архангельская область без  авт. округа',
    'Тюменская область без авт.округов': 'Тюменская область (без авт. округов)',
    'Ненецкий авт.округ': 'Ненецкий автономный округ',
    'Республика Северная Осетия - Алания': 'Республика Северная Осетия-Алания',
    'Ханты-Мансийский авт.округ': 'Ханты-Мансийский автономный округ - Югра',
    'Еврейская авт.область': 'Еврейская автономная область',
    'Ямало-Ненецкий авт.округ': 'Ямало-Ненецкий автономный округ',
    'Чукотский авт.округ': 'Чукотский автономный округ'
})


In [13]:
merged = poverty.merge(poverty_line, on='region', how='left').merge(income, on='region', how='outer')

In [14]:
merged = merged[~merged['region'].str.contains('федеральный округ', na=False)]
merged = merged[~merged['region'].isin(['Архангельская область', 'Тюменская область'])]

In [15]:
merged = merged.merge(regions_codes, on='region', how='left')
merged = merged.merge(basket[['region', 'average', 'rate']], on='region', how='left')

In [16]:
merged

,region,poverty,poverty_line,income,region_code,region_short_name,average,rate
0,Алтайский край,12.1,14432,42721,01,Алтайский край,22539.225000,0.998824
1,Амурская область,9.5,18352,64337,10,Амурская обл.,23677.184167,1.049253
2,Архангельская область без авт. округа,8.1,18639,61323,11,Архангельская обл.,24799.099167,1.098970
3,Астраханская область,10.7,14724,41393,12,Астраханская обл.,19447.075833,0.861796
4,Белгородская область,4.4,13257,56742,14,Белгородская обл.,19544.085000,0.866095
...,...,...,...,...,...,...,...,...
81,Чеченская Республика,15.2,15126,40474,96,Чечня,20571.445000,0.911622
82,Чувашская Республика,10.4,13565,39362,97,Чувашия,19486.488333,0.863542
83,Чукотский автономный округ,4.4,28813,165034,77,Чукотский АО,36570.944167,1.620639
84,Ямало-Ненецкий автономный округ,3.5,21151,153627,74,ЯНАО,26486.118333,1.173731


## 2. Расчет распределения доходов

In [17]:
from hack_params import HackParams

In [18]:
merged['income'] = merged['income'].astype(float)
merged['poverty'] = merged['poverty'].astype(float)
merged['poverty_line'] = merged['poverty_line'].astype(float)

In [19]:
poverty_steps_all = []

for idx, row in tqdm(merged.iterrows(), total=len(merged)):
    new_poverty = HackParams(average_income=row['income'], 
                             poverty_line=row['poverty_line'], 
                             poverty=row['poverty'])

    new_poverty.find_params()

    poverty_steps = pd.DataFrame({'sum': list(range(1000, 500001, 500))})
    poverty_steps['sum_corr'] = poverty_steps['sum']/row['rate']
    poverty_steps['share'] = poverty_steps['sum'].apply(new_poverty.count_poverty)
    poverty_steps['step_share'] = poverty_steps['share'] - poverty_steps['share'].shift(fill_value=0)

    last_share = poverty_steps['share'].iloc[-1]
    new_row = {
        'sum': 500050,
        'sum_corr': 500050/row['rate'],
        'share': 100,
        'step_share': 100 - last_share,
        'region': row['region'],
        'region_code': row['region_code'],
        'region_short_name': row['region_short_name'],
        'poverty': row['poverty'],
        'poverty_line': row['poverty_line'],
        'income': row['income']
    }
    poverty_steps = pd.concat([poverty_steps, pd.DataFrame([new_row])], ignore_index=True)

    poverty_steps['region'] = row['region']
    poverty_steps['region_code'] = row['region_code']
    poverty_steps['region_short_name'] = row['region_short_name']
    poverty_steps['poverty'] = row['poverty']
    poverty_steps['poverty_line'] = row['poverty_line']
    poverty_steps['income'] = row['income']
    poverty_steps['average'] = row['average']   
    poverty_steps['rate'] = row['rate']

    poverty_steps_all.append(poverty_steps)

poverty_steps_df = pd.concat(poverty_steps_all, ignore_index=True)

100%|██████████| 86/86 [00:25<00:00,  3.41it/s]


In [20]:
poverty_steps_df.to_excel('./data/poverty_steps_df_500_v20251216.xlsx', index=False)
poverty_steps_df.to_csv('./data/poverty_steps_df_500_v20251216.csv', index=False)

In [27]:
poverty_steps_df.head()

,sum,sum_corr,share,step_share,region,region_code,region_short_name,poverty,poverty_line,income,average,rate
0,1000,1001.177170,0.0,0.0,Алтайский край,01,Алтайский край,12.1,14432.0,42721.0,22539.225,0.998824
1,1050,1051.236029,0.0,0.0,Алтайский край,01,Алтайский край,12.1,14432.0,42721.0,22539.225,0.998824
2,1100,1101.294887,0.0,0.0,Алтайский край,01,Алтайский край,12.1,14432.0,42721.0,22539.225,0.998824
3,1150,1151.353746,0.0,0.0,Алтайский край,01,Алтайский край,12.1,14432.0,42721.0,22539.225,0.998824
4,1200,1201.412604,0.0,0.0,Алтайский край,01,Алтайский край,12.1,14432.0,42721.0,22539.225,0.998824


In [22]:
data = pd.read_csv('/Users/vkopytok/Downloads/votes.csv')

In [26]:
data.to_excel('/Users/vkopytok/Downloads/votes.xlsx', index=False)

In [ ]:
data = data[data['is_main'] == True]